In [1]:
import pandas as pd
import numpy as np
import re, os, joblib, warnings

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack

warnings.filterwarnings("ignore")

In [4]:
df_full = pd.read_excel('./data/sinhala_biz_news.xlsx')
df_full["news_content"] = df_full["news_content"].fillna("")

companies = [
    'WATA.N0000','AGPL.N0000','HAPU.N0000','MCPL.N0000',
    'KOTA.N0000','BFL.N0000','RWSL.N0000','DIPP.N0000',
    'MGT.N0000','HEXP.N0000'
]

# Use only labelled rows
df = df_full.head(4352).copy()
df[companies] = df[companies].fillna("Neutral")

X_raw = df_full["news_content"]

print("Train rows:", len(df))
print("Corpus size:", len(X_raw))

Train rows: 4352
Corpus size: 8051


In [5]:
df.head()

,Unnamed: 0,date,headline,news_content,MCPL.N0000,KOTA.N0000,HAPU.N0000,WATA.N0000,AGPL.N0000,BFL.N0000,RWSL.N0000,DIPP.N0000,MGT.N0000,HEXP.N0000,Unnamed: 14,Unnamed: 15,Unnamed: 16,url
0,1,2025-09-04 06:11:18,ඩිජිටල් සේවා සඳහා වැට් බද්ද බලාත්මක කිරීම කල් යයි,2025 අංක 4 දරන එකතු කළ අගය මත බදු (සංශෝධන) පනත...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b6%a9%e0%b7%92%e0%...
1,2,NaN,පාඩු ලබන රාජ්‍ය ව්‍යවසාය 33ක් වසා දැමීමට කැබින...,මහජන සේවා සැපයීම සහ උපායමාර්ගික ආර්ථික ක්‍රියා...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b6%b4%e0%b7%8f%e0%...
2,3,2025-09-03 10:53:35,මෙරට නිෂ්පාදිත ඖෂධ ලෝක වෙළඳපොලට,• 100%ක දේශීය සමාගමක් ඖෂධ අපනයනයට එක් වූ මුල් ...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b6%b8%e0%b7%99%e0%...
3,4,2025-09-03 10:40:46,ශ්‍රී ලංකාවේ ප්‍රථම විදුලි සංදේශ යටිතල පහසුකම්...,EDOTCO ආයතනයට ශ්‍රී ලංකාවේ ප්‍රථම විදුලි සංදේශ...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b7%81%e0%b7%8a%e2%...
4,5,2025-09-03 06:46:31,සංචාරක ක්ෂේත්‍රය ප්‍රවර්ධනය කිරීමේ අරමුණින් කා...,සංචාරක ක්ෂේත්‍රය ප්‍රවර්ධනය කිරීමේ අරමුණින් කා...,Neutral,Neutral,Neutral,Neutral,Neutral,Positive,Positive,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b7%83%e0%b6%82%e0%...


In [6]:
stop_set = set(open('./data/stop words.txt', encoding='utf-8').read().splitlines())

def word_tokenizer(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    return [t for t in tokens if t not in stop_set and len(t) > 1]

In [7]:
print("Fitting TF-IDF...")

tfidf_word = TfidfVectorizer(
    tokenizer=word_tokenizer,
    token_pattern=None,
    max_features=8000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    max_features=8000,
    ngram_range=(2,4),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

X_word = tfidf_word.fit_transform(X_raw)
X_char = tfidf_char.fit_transform(X_raw)

X_all = hstack([X_word, X_char])

print("TF-IDF Shape:", X_all.shape)

Fitting TF-IDF...
TF-IDF Shape: (8051, 16000)


In [8]:
def apply_thresholds(proba, classes, thresholds):
    preds = []
    for row in proba:
        chosen = None
        best_score = -1

        for i, cls in enumerate(classes):
            if cls != 'Neutral' and row[i] >= thresholds.get(cls, 0.5):
                if row[i] > best_score:
                    best_score = row[i]
                    chosen = cls

        if chosen is None:
            chosen = classes[np.argmax(row)]

        preds.append(chosen)

    return np.array(preds)

In [9]:
def tune_thresholds(clf, X_tr, y_tr, classes):
    oof = cross_val_predict(clf, X_tr, y_tr, cv=5, method='predict_proba')

    thresholds = {}

    for i, cls in enumerate(classes):
        if cls == 'Neutral':
            thresholds[cls] = 0.5
            continue

        best_th, best_f1 = 0.5, 0

        for th in np.arange(0.08, 0.55, 0.01):
            preds = []

            for row in oof:
                if row[i] >= th:
                    preds.append(cls)
                else:
                    preds.append(classes[np.argmax(row)])

            f1 = f1_score(y_tr, preds, labels=[cls],
                          average='macro', zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_th = th

        thresholds[cls] = best_th
        print(f"{cls}: threshold={best_th:.2f}, F1={best_f1:.3f}")

    return thresholds

In [10]:
os.makedirs("saved_models_improved", exist_ok=True)

results = []

for company in companies:

    print(f"\n===== {company} =====")

    y_all = df[company].astype(str).values
    idx = np.arange(len(y_all))

    idx_tr, idx_te, y_train, y_test = train_test_split(
        idx, y_all,
        test_size=0.2,
        random_state=42,
        stratify=y_all
    )

    X_tr = X_all[idx_tr]
    X_te = X_all[idx_te]

    # Class distribution
    vc = pd.Series(y_train).value_counts()
    neutral_n = vc.get('Neutral', 0)
    pos_n = vc.get('Positive', 0)
    neg_n = vc.get('Negative', 0)

    print("Train dist:", dict(vc))


===== WATA.N0000 =====
Train dist: {'Neutral': np.int64(2515), 'Positive': np.int64(676), 'Negative': np.int64(290)}

===== AGPL.N0000 =====
Train dist: {'Neutral': np.int64(2528), 'Positive': np.int64(665), 'Negative': np.int64(288)}

===== HAPU.N0000 =====
Train dist: {'Neutral': np.int64(2522), 'Positive': np.int64(666), 'Negative': np.int64(293)}

===== MCPL.N0000 =====
Train dist: {'Neutral': np.int64(2548), 'Positive': np.int64(647), 'Negative': np.int64(286)}

===== KOTA.N0000 =====
Train dist: {'Neutral': np.int64(2520), 'Positive': np.int64(667), 'Negative': np.int64(294)}

===== BFL.N0000 =====
Train dist: {'Neutral': np.int64(2561), 'Positive': np.int64(670), 'Negative': np.int64(250)}

===== RWSL.N0000 =====
Train dist: {'Neutral': np.int64(2682), 'Positive': np.int64(590), 'Negative': np.int64(209)}

===== DIPP.N0000 =====
Train dist: {'Neutral': np.int64(2518), 'Positive': np.int64(671), 'Negative': np.int64(292)}

===== MGT.N0000 =====
Train dist: {'Neutral': np.int64(2

In [11]:
    target = neutral_n // 2
    strategy = {}

    if pos_n < target:
        strategy['Positive'] = target
    if neg_n < target:
        strategy['Negative'] = target

    if strategy:
        le = LabelEncoder().fit(y_train)
        y_enc = le.transform(y_train)

        int_strategy = {
            le.transform([cls])[0]: n
            for cls, n in strategy.items()
        }

        k = max(1, min(5, neg_n // 3))

        smote = SMOTE(
            sampling_strategy=int_strategy,
            random_state=42,
            k_neighbors=k
        )

        X_tr_fit, y_fit_enc = smote.fit_resample(X_tr, y_enc)
        y_tr_fit = le.inverse_transform(y_fit_enc)

        print("After SMOTE:", pd.Series(y_tr_fit).value_counts().to_dict())
    else:
        X_tr_fit = X_tr
        y_tr_fit = y_train

After SMOTE: {'Neutral': 2529, 'Positive': 1264, 'Negative': 1264}


In [12]:
    best_model = None
    best_score = -999

    for C in [0.05, 0.1, 0.3, 0.5, 1.0]:

        base = LinearSVC(
            class_weight='balanced',
            C=C,
            max_iter=3000
        )

        clf = CalibratedClassifierCV(base, cv=3, method='isotonic')
        clf.fit(X_tr_fit, y_tr_fit)

        classes = list(clf.classes_)

        oof = cross_val_predict(
            clf, X_tr_fit, y_tr_fit,
            cv=3, method='predict_proba'
        )

        preds = [classes[np.argmax(r)] for r in oof]

        f1 = f1_score(y_tr_fit, preds, average='macro', zero_division=0)

        if f1 > best_score:
            best_score = f1
            best_model = clf

In [13]:
    print("Tuning thresholds...")
    thresholds = tune_thresholds(best_model, X_tr_fit, y_tr_fit, classes)

Tuning thresholds...
Negative: threshold=0.45, F1=0.965
Positive: threshold=0.44, F1=0.834


In [14]:
    tr_preds = apply_thresholds(best_model.predict_proba(X_tr), classes, thresholds)
    te_preds = apply_thresholds(best_model.predict_proba(X_te), classes, thresholds)

    print("Train Acc:", accuracy_score(y_train, tr_preds))
    print("Test Acc :", accuracy_score(y_test, te_preds))

    print("\nClassification Report:")
    print(classification_report(y_test, te_preds))

Train Acc: 0.9870726802642918
Test Acc : 0.8071182548794489

Classification Report:
              precision    recall  f1-score   support

    Negative       0.67      0.39      0.50        71
     Neutral       0.86      0.91      0.89       633
    Positive       0.60      0.57      0.59       167

    accuracy                           0.81       871
   macro avg       0.71      0.63      0.66       871
weighted avg       0.80      0.81      0.80       871



In [18]:
# ==========================================================
# COST-SENSITIVE SVM (FINAL CLEAN VERSION)
# TRAIN + SAVE + PREDICT (ALL 10 COMPANIES)
# ==========================================================

import pandas as pd
import re
import os
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings("ignore")

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_excel('./data/sinhala_biz_news.xlsx').head(4351)
df['news_content'] = df['news_content'].fillna('')

companies = [
    'WATA.N0000','AGPL.N0000','HAPU.N0000','MCPL.N0000',
    'KOTA.N0000','BFL.N0000','RWSL.N0000','DIPP.N0000',
    'MGT.N0000','HEXP.N0000'
]

df[companies] = df[companies].fillna('Neutral')

X = df['news_content']

# ==========================================================
# STOPWORDS + TOKENIZER
# ==========================================================

stop_words = set(open('./data/stop words.txt', encoding='utf-8').read().splitlines())

def tokenizer(text):
    tokens = re.findall(r'\b\w+\b', str(text).lower())
    return [t for t in tokens if t not in stop_words and len(t) > 1]

# ==========================================================
# CLASS WEIGHTS (COST-SENSITIVE)
# ==========================================================

class_weights = {
    'Neutral': 1,
    'Positive': 2,
    'Negative': 3
}

# ==========================================================
# CREATE MODEL DIRECTORY
# ==========================================================

os.makedirs("saved_models", exist_ok=True)

# ==========================================================
# TRAIN + SAVE MODELS (ALL 10 COMPANIES)
# ==========================================================

results = []

for company in companies:

    print(f"\n{'='*50}")
    print(f"TRAINING MODEL FOR: {company}")
    print(f"{'='*50}")

    y = df[company].astype(str)

    # PIPELINE
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            tokenizer=tokenizer,
            token_pattern=None,      # removes sklearn warning
            max_features=1000,
            ngram_range=(1,2)        # better performance
        )),
        ('svm', SVC(
            kernel='linear',
            class_weight=class_weights,
            probability=True         # enables future probability usage
        ))
    ])

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predictions
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # Accuracy
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test, y_test_pred)

    print("Training Accuracy:", round(train_acc, 4))
    print("Testing Accuracy :", round(test_acc, 4))

    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred, digits=3, zero_division=0))

    # Save model
    model_path = f"saved_models/{company}_cost_sensitive_svm.joblib"
    joblib.dump(pipeline, model_path)

    print(f"Model saved → {model_path}")

    results.append({
        'Company': company,
        'Train Acc': round(train_acc, 4),
        'Test Acc': round(test_acc, 4)
    })

# ==========================================================
# SAVE SUMMARY
# ==========================================================

accuracy_df = pd.DataFrame(results)
accuracy_df.to_csv("cost_sensitive_svm_accuracy.csv", index=False)

print("\n===== FINAL SUMMARY =====")
print(accuracy_df.to_string(index=False))

# ==========================================================
# LOAD ALL MODELS ONCE (FASTER PREDICTION)
# ==========================================================

loaded_models = {
    company: joblib.load(f"saved_models/{company}_cost_sensitive_svm.joblib")
    for company in companies
}

# ==========================================================
# PREDICTION FUNCTIONS
# ==========================================================

def predict_sentiment(news_text, company):
    return loaded_models[company].predict([news_text])[0]

def predict_all_companies(news_text):
    return {
        company: predict_sentiment(news_text, company)
        for company in companies
    }

# ==========================================================
# EXAMPLE TEST
# ==========================================================

if __name__ == "__main__":

    latest_news = """
    Hayleys Fabric සමාගම විසින් මෙම වසරේ ලාභය 40%කින් ඉහළ යාමක් වාර්තා කර ඇත.
    නව ව්‍යාපෘති සහ ආයෝජන හේතුවෙන් ආදායම වර්ධනය වී ඇත.
    """

    print("\n===== PREDICTIONS =====")

    predictions = predict_all_companies(latest_news)

    for company, sentiment in predictions.items():
        print(f"{company} → {sentiment}")


TRAINING MODEL FOR: WATA.N0000
Training Accuracy: 0.8667
Testing Accuracy : 0.7879

Classification Report:
              precision    recall  f1-score   support

    Negative      0.538     0.587     0.561       109
     Neutral      0.876     0.880     0.878       943
    Positive      0.565     0.531     0.548       254

    accuracy                          0.788      1306
   macro avg      0.659     0.666     0.662      1306
weighted avg      0.787     0.788     0.787      1306

Model saved → saved_models/WATA.N0000_cost_sensitive_svm.joblib

TRAINING MODEL FOR: AGPL.N0000
Training Accuracy: 0.8719
Testing Accuracy : 0.7833

Classification Report:
              precision    recall  f1-score   support

    Negative      0.484     0.556     0.517       108
     Neutral      0.878     0.881     0.880       949
    Positive      0.552     0.510     0.530       249

    accuracy                          0.783      1306
   macro avg      0.638     0.649     0.642      1306
weighted avg 